# NemotronRelational Quickstart with the NVIDIA Nemotron Predict Client

**NemotronRelational (Nemotron Relational Relational Foundation Model)** is a foundation model for
machine learning on enterprise data. With your relational data and a
Predictive Query Language (PQL) query, you can generate predictions without
training a task-specific model.

This quickstart uses four tables from Microsoft's **AdventureWorks** OLTP
sample database: customers, products, sales-order headers, and sales-order
details. Predictions are submitted to a deployed NemotronRelational NVIDIA NIM through
the NVIDIA Nemotron Predict Client.


## Introduction

Nemotron Relational is grounded in three key world views:

> **1. Enterprise data is a graph.**

Enterprise data is a graph where tables are connected by keys. In this notebook, `sales_order_headers` connects customers to orders, while `sales_order_details` connects each order to its products. Together, these relationships give Nemotron Relational the business context surrounding every transaction.

Once we structure enterprise data this way, we can apply pre-trained Relational Graph Transformers to extract insights and patterns.

> **2. With timestamps, we place events on a timeline.**

By placing events on a timeline, we unlock the ability to model how things evolve. We can select any point in time and predict what is likely to happen next from the sequence and patterns in the historical data.

> **3. Machine learning tasks can be described via predictive queries.**

Major machine learning tasks, including regression, classification, forecasting, and recommendation, can be expressed using *Predictive Query Language (PQL)*. A PQL query identifies the outcome to predict, the entity to predict it for, and, for temporal tasks, the future prediction window.

If you know SQL, PQL should feel familiar. Learn more in the [PQL documentation](https://docs.nvidia.com/sdgm/reference/pq-reference-overview).


**Let's get started!**



## Step 1. Install the NVIDIA Nemotron Predict Client

Install the client with the Nemotron Relational extra. The Nemotron Relational release wheels support
Python 3.10, 3.11, 3.12, and 3.13.


In [ ]:
%pip install "nemotron-predict-client[relational]"

In [ ]:
import os
from pathlib import Path

import pandas as pd
from nemotron_predict import PredictClient, relational

The supported Nemotron Relational object namespace is
`nemotron_predict.relational`. Build graphs through that namespace and submit
predictions through `PredictClient`.


## Step 2. Configure the NemotronRelational NIM endpoint

Deploy Nemotron Relational as an NVIDIA NIM before running this notebook. See
[Deploy, Install, and Connect](https://docs.nvidia.com/sdgm/rfm/sdk-getting-started)
for deployment instructions.

Set `NEMOTRON_PREDICT_API_ENDPOINT` to the reachable NIM endpoint. The default below uses a
local container on port `8000`. Set `NEMOTRON_PREDICT_API_KEY` only when an
authenticating gateway protects the endpoint.


In [ ]:
NIM_URL = os.environ.get(
    'NEMOTRON_PREDICT_API_ENDPOINT',
    'http://localhost:8000',
)
NIM_API_KEY = os.environ.get('NEMOTRON_PREDICT_API_KEY')

## Step 3. Connect to the NIM

An `PredictClient` owns the HTTP connection to one NIM endpoint. Keep credentials
in environment configuration rather than notebook source.


In [ ]:
client = PredictClient(
    url=NIM_URL,
    api_key=NIM_API_KEY,
)

print('Ready:', client.health_ready())
print('Capabilities:', client.capabilities('nemotron-relational'))

<div class="alert alert-warning">
<strong>Security:</strong> Never hard-code NIM or gateway credentials in a
notebook. Use environment variables or your platform's secret manager.
</div>


## Step 4. Data Preparation

This notebook uses the **AdventureWorks OLTP sample database**, originally
published by Microsoft as a SQL Server sample. The four source extracts are:

- `Customer.csv`
- `Product.csv`
- `SalesOrderHeader.csv`
- `SalesOrderDetail.csv`

The notebook downloads the four `*.csv` extracts from the AdventureWorks OLTP install scripts in the
[Microsoft SQL Server samples repository](https://github.com/microsoft/sql-server-samples/tree/master/samples/databases/adventure-works/oltp-install-script)
when they are not already present in the data directory.

> Each user is responsible for checking the content of datasets and the
> applicable licenses and determining if suitable for the intended use.
The repository's samples and templates are provided under the
[MIT License](https://github.com/microsoft/sql-server-samples/blob/master/license.txt).

Set `ADVENTURE_WORKS_DATA_DIR` to choose where the files are read from and cached,
e.g. `export ADVENTURE_WORKS_DATA_DIR=/path/to/adventure-works-cache`. If it is unset,
the notebook uses `../rfm/adventure-works` relative to its working directory.
The files are tab-delimited and do not include a header row, so each schema is
assigned explicitly when the file is loaded.

The sales-order header and detail tables remain separate. During preparation,
only `CustomerID` and `OrderDate` are copied from each header to its detail
rows using `SalesOrderID`.


In [ ]:
import urllib.request
from pathlib import Path

import pandas as pd

SOURCE_BASE = (
    'https://raw.githubusercontent.com/microsoft/sql-server-samples'
    '/master/samples/databases/adventure-works/oltp-install-script'
)

DATA_DIR = (
    Path(os.environ.get('ADVENTURE_WORKS_DATA_DIR', '../rfm/adventure-works'))
    .expanduser()
    .resolve()
)

print('AdventureWorks data:', DATA_DIR)

CUSTOMER_COLUMNS = [
    'CustomerID',
    'PersonID',
    'StoreID',
    'TerritoryID',
    'AccountNumber',
    'rowguid',
    'ModifiedDate',
]

PRODUCT_COLUMNS = [
    'ProductID',
    'Name',
    'ProductNumber',
    'MakeFlag',
    'FinishedGoodsFlag',
    'Color',
    'SafetyStockLevel',
    'ReorderPoint',
    'StandardCost',
    'ListPrice',
    'Size',
    'SizeUnitMeasureCode',
    'WeightUnitMeasureCode',
    'Weight',
    'DaysToManufacture',
    'ProductLine',
    'Class',
    'Style',
    'ProductSubcategoryID',
    'ProductModelID',
    'SellStartDate',
    'SellEndDate',
    'DiscontinuedDate',
    'rowguid',
    'ModifiedDate',
]

SALES_HEADER_COLUMNS = [
    'SalesOrderID',
    'RevisionNumber',
    'OrderDate',
    'DueDate',
    'ShipDate',
    'Status',
    'OnlineOrderFlag',
    'SalesOrderNumber',
    'PurchaseOrderNumber',
    'AccountNumber',
    'CustomerID',
    'SalesPersonID',
    'TerritoryID',
    'BillToAddressID',
    'ShipToAddressID',
    'ShipMethodID',
    'CreditCardID',
    'CreditCardApprovalCode',
    'CurrencyRateID',
    'SubTotal',
    'TaxAmt',
    'Freight',
    'TotalDue',
    'Comment',
    'rowguid',
    'ModifiedDate',
]

SALES_DETAIL_COLUMNS = [
    'SalesOrderID',
    'SalesOrderDetailID',
    'CarrierTrackingNumber',
    'OrderQty',
    'ProductID',
    'SpecialOfferID',
    'UnitPrice',
    'UnitPriceDiscount',
    'LineTotal',
    'rowguid',
    'ModifiedDate',
]


def read_headerless_table(filename, columns):
    path = DATA_DIR / filename
    if not path.exists():
        # Fetched from the upstream AdventureWorks sample rather than kept in
        # the repository, so a fresh clone runs without a manual download.
        DATA_DIR.mkdir(parents=True, exist_ok=True)
        print(f'downloading {filename}')
        urllib.request.urlretrieve(f'{SOURCE_BASE}/{filename}', path)
    return pd.read_csv(
        path,
        sep='\t',
        header=None,
        names=columns,
        na_values=[''],
        low_memory=False,
    )


customers_df = read_headerless_table('Customer.csv', CUSTOMER_COLUMNS)
products_df = read_headerless_table('Product.csv', PRODUCT_COLUMNS)
sales_order_headers_df = read_headerless_table(
    'SalesOrderHeader.csv', SALES_HEADER_COLUMNS
)
sales_order_details_df = read_headerless_table(
    'SalesOrderDetail.csv', SALES_DETAIL_COLUMNS
)

Parse the timestamp fields, then enrich each sales-order detail with only its parent order's `CustomerID` and `OrderDate`. The header and detail dataframes remain separate.


In [ ]:
timestamp_columns = {
    'customers': (customers_df, ['ModifiedDate']),
    'products': (
        products_df,
        ['SellStartDate', 'SellEndDate', 'DiscontinuedDate', 'ModifiedDate'],
    ),
    'sales_order_headers': (
        sales_order_headers_df,
        ['OrderDate', 'DueDate', 'ShipDate', 'ModifiedDate'],
    ),
    'sales_order_details': (sales_order_details_df, ['ModifiedDate']),
}

for _, (table_df, columns) in timestamp_columns.items():
    for column in columns:
        table_df[column] = pd.to_datetime(table_df[column], errors='coerce')

sales_order_details_df = sales_order_details_df.merge(
    sales_order_headers_df[['SalesOrderID', 'CustomerID', 'OrderDate']],
    on='SalesOrderID',
    how='left',
    validate='many_to_one',
)

## Step 5. Build a graph from DataFrames

`relational.Graph.from_data()` accepts a dictionary of pandas DataFrames and
creates the underlying local table objects automatically. You do not need to
construct `LocalTable` objects for the standard workflow.

The AdventureWorks relationships are supplied explicitly so the example is
reproducible. Nemotron Relational still infers column data types and semantic metadata,
which you should review before prediction.


Create the graph directly from the four prepared
DataFrames:


In [ ]:
graph = relational.Graph.from_data(
    {
        'customers': customers_df,
        'products': products_df,
        'sales_order_headers': sales_order_headers_df,
        'sales_order_details': sales_order_details_df,
    },
    edges=[
        ('sales_order_headers', 'CustomerID', 'customers'),
        ('sales_order_details', 'SalesOrderID', 'sales_order_headers'),
        ('sales_order_details', 'ProductID', 'products'),
        ('sales_order_details', 'CustomerID', 'customers'),
    ],
    verbose=False,
)

Review the inferred metadata before making any changes.

The graph-level view lists every table with its primary key and time columns.
The table-level views show each column's data type (`dtype`), semantic type
(`stype`), and key or time-column role.

In [ ]:
# Review the inferred graph and table definitions.
graph.print_metadata()

for table in graph.tables.values():
    table.print_metadata()

If the inferred definition does not match your schema, edit only the
metadata that needs correction:

- Set a primary key with `graph["table_name"].primary_key = "column_name"`.
  The client automatically assigns the `ID` semantic type to that column.
- Set the event-time column with
  `graph["table_name"].time_column = "column_name"`.
- Override one column's semantic type with
  `graph["table_name"]["column_name"].stype = nemotron_relational.Stype.<type>`.

Use the `nemotron_relational.Stype` enum so the intended type is explicit. The client checks
that the semantic type is compatible with the column's data type. Foreign-key
columns used in the `edges` argument are already assigned `Stype.ID`.

For this known AdventureWorks schema, review identifier-like strings before
prediction. Account numbers, product numbers, order numbers, and approval codes
identify records rather than categories, so the example marks them `Stype.ID`.
Several exceed the NIM's 10,000-value categorical cardinality limit in the full
extracts. Product names remain `Stype.text` because their language content is a
useful feature, while order quantity is a measured numerical value.


In [ ]:
# Correct primary keys that are specific to the known schema.
# Assigning a primary key also assigns Stype.ID to that column.
primary_keys = {
    'customers': 'CustomerID',
    'products': 'ProductID',
    'sales_order_headers': 'SalesOrderID',
    'sales_order_details': 'SalesOrderDetailID',
}

for table_name, column_name in primary_keys.items():
    graph[table_name].primary_key = column_name

# Identify the event-time column for tables that contain timestamped events.
graph['sales_order_headers'].time_column = 'OrderDate'
graph['sales_order_details'].time_column = 'OrderDate'

# Correct semantic types that are specific to this known schema. These values
# identify business records rather than categories or free-text documents.
id_columns = {
    'customers': ['AccountNumber'],
    'products': ['ProductNumber'],
    'sales_order_headers': [
        'SalesOrderNumber',
        'PurchaseOrderNumber',
        'AccountNumber',
        'CreditCardApprovalCode',
    ],
}

for table_name, column_names in id_columns.items():
    for column_name in column_names:
        graph[table_name][column_name].stype = relational.Stype.ID

# Product names carry natural-language meaning; order quantity is measured.
graph['products']['Name'].stype = relational.Stype.text
graph['sales_order_details']['OrderQty'].stype = relational.Stype.numerical

> **Important:** A column can have a datetime data type and the
> `timestamp` semantic type without being the table's designated
> `time_column`. Set `time_column` only when the column represents the table's
> event timeline, the time Nemotron Relational should use to order events and construct
> temporal context. Administrative timestamps such as `ModifiedDate` do not
> necessarily serve that role.

In [ ]:
# These tables contain datetime columns but do not represent event timelines.
graph['customers'].time_column = None
graph['products'].time_column = None

Review the updated definition to confirm that the corrections were
applied:

In [ ]:
graph.print_metadata()

for table in graph.tables.values():
    table.print_metadata()

Primary-key assignment automatically gives each key column the `ID`
semantic type. Relationships supplied through `edges` do the same for their
foreign-key columns, so neither needs a separate semantic-type override.

Other available semantic types include `categorical`, `multicategorical`,
`numerical`, `text`, `timestamp`, and `unsupported`.

Call `graph.validate()` after all metadata changes. An incompatible semantic
type, a missing column, or a primary key that is also configured as a time
column raises an error.

In [ ]:
graph.validate()

**Quick reference:**

1. **`stype` (semantic type)**

   A semantic type determines how a column is encoded downstream.

   | Type | Explanation | AdventureWorks example |
   |---|---|---|
   | `"numerical"` | A measurable numeric value | `LineTotal`, `OrderQty` |
   | `"categorical"` | A value from a limited set of categories | `Color`, `ProductLine` |
   | `"multicategorical"` | Multiple categories stored in one cell | A pipe-delimited tag column |
   | `"ID"` | A primary key, foreign key, or other identifier | `CustomerID`, `ProductID` |
   | `"text"` | Natural-language text | A product description |
   | `"timestamp"` | A point in time | `OrderDate` |
   | `"sequence"` | Custom embeddings or sequential data | A numeric embedding vector |

2. **`primary_key`**

   - A primary key uniquely identifies each row.
   - Duplicate primary-key values are not supported as distinct entities.
   - A primary key can be the destination of a primary-key/foreign-key link.
   - Each table can have at most one primary-key column.

3. **`time_column`**

   - A time column records when an event occurred.
   - Its values must be parseable by `pandas.to_datetime`.
   - Each table can have at most one time column.


## Step 6. Inspect the relational graph

The graph preserves the four separate AdventureWorks tables:

- each sales-order header belongs to one customer;
- each sales-order detail belongs to one sales-order header;
- each sales-order detail references one product; and
- the copied `CustomerID` links each detail row to its customer.

Review the registered links before sending prediction requests.


Print the relationships supplied to
`Graph.from_data()`:


In [ ]:
graph.print_links()

Visualize the graph as a Mermaid
entity-relationship diagram:


In [ ]:
graph.visualize()

Validate the completed graph:


In [ ]:
graph.validate()

## Step 7. Submit prediction requests

The graph is a reusable description of the data. For each prediction:

1. Write the task in PQL.
2. Bind the graph to a model handle with `client.relational(graph)`.
3. Call `.predict(query, indices=[...])` on that handle.

The client returns a pandas DataFrame. Standard applications do not initialize
the underlying Nemotron Relational driver or model object directly.


**Note:** AdventureWorks is sample data, and the queries and results below are intended for demonstration. Benchmark Nemotron Relational on data representative of your own use case before drawing performance conclusions.


#### PQL: How you describe a prediction task

Learn more in the
[PQL documentation](https://docs.nvidia.com/sdgm/reference/pq-reference-overview)
and [query-writing guide](https://docs.nvidia.com/sdgm/rfm/writing-predictive-queries).

The short reference below covers the entity, target, aggregation,
prediction-window, and condition syntax used by the examples in this
quickstart.


<details>
<summary><b>💡 Click here for a short introduction to Predictive Query!</b></summary>

**Entities** can be specified in three ways:

* `FOR <entity_table>.<entity_primary_key>=1` embeds one entity ID in PQL.
* `FOR <entity_table>.<entity_primary_key> IN (1, 2, 3)` embeds several
  entity IDs in PQL.
* `FOR EACH <entity_table>.<entity_primary_key>` declares the entity column,
  while `predict(query, indices=[1, 2, 3])` supplies the entity IDs separately.

**Preferred:** For application code, use `FOR EACH` and pass primary-key values
through `predict()`'s `indices` argument. This keeps the prediction task separate
from the entity batch and makes the same PQL reusable. Despite its name,
`indices` contains source primary-key values, not DataFrame row numbers or
internal graph indices. If `indices` is provided, it overrides entity IDs
embedded in the PQL.

**Targets** can be specified in two common ways:

* `PREDICT <target_table>.<target_column>` imputes a missing value.
* `PREDICT <aggregation>(<target>, <start_offset>, <end_offset>, <time_unit>)` predicts an aggregate in a future window.

Supported aggregations include `COUNT`, `SUM`, `AVG`, `MIN`, `MAX`, and `LIST_DISTINCT`. For example, `PREDICT COUNT(sales_order_headers.*, 0, 7, days)` predicts the number of orders in the next seven days.

Conditions can turn an aggregate into an event. For example, appending `=0` predicts the probability that no orders occur in the specified window.
</details>


### Example 1A: Forecast 30-day product demand

Predict how many units of product `ProductID=870` will be ordered in the next 30 days.


In [ ]:
query = (
    'PREDICT SUM(sales_order_details.OrderQty, 0, 30, days) '
    'FOR EACH products.ProductID'
)

result = client.relational(graph).predict(
    query,
    indices=[870],
    run_mode='fast',
)
display(result)

How to interpret the result:

1. `ENTITY`: the product with `ProductID=870`
2. `ANCHOR_TIMESTAMP`: the point in time from which the next 30 days are predicted
3. `PREDICTION`: the predicted number of units ordered during that window

This result can support inventory and demand planning.


### Example 1B: Forecast from a historical anchor time

By default, predictions use the maximum timestamp in the temporal graph. Set a historical `anchor_time` to simulate what the prediction would have looked like at that point in time. Nemotron Relational uses only information available before the anchor time.


In [ ]:
# Pin the anchor instead of letting it be derived. It is taken from the
# data rather than written as a literal: AdventureWorks is re-dated
# upstream, so a hardcoded date stops matching the file you just read.
anchor_time = sales_order_headers_df['OrderDate'].max() - pd.Timedelta(days=60)

result = client.relational(graph).predict(
    query,
    indices=[870],
    run_mode='fast',
    anchor_time=anchor_time,
)
display(result)

### Example 1C: Forecast for multiple steps

In [ ]:
query = (
    'PREDICT SUM(sales_order_details.OrderQty, 0, 30, days) '
    'FORECAST 3 TIMEFRAMES FOR EACH products.ProductID'
)

result = client.relational(graph).predict(
    query,
    indices=[870],
    run_mode='fast',
)
display(result)

A multi-step forecast returns one row per timeframe.
`forecast_step` is the one-based sequence number and `PREDICTION` is the
forecast value for that period.


### Example 2: Predict customer inactivity

Predict the likelihood that customers `CustomerID=11176` and `CustomerID=11091` will place zero orders in the next 90 days.


In [ ]:
query = (
    'PREDICT COUNT(sales_order_headers.*, 0, 90, days)=0 '
    'FOR EACH customers.CustomerID'
)

result = client.relational(graph).predict(
    query,
    indices=[11176, 11091],
    run_mode='fast',
)
display(result)

How to interpret the result:

1. `ENTITY`: the customer represented by each `CustomerID`
2. `ANCHOR_TIMESTAMP`: the point in time from which the next 90 days are predicted
3. `PREDICTION`: whether the zero-order event is predicted to occur
4. Probability columns: the estimated likelihood of each outcome

This result can support retention or re-engagement workflows.


### Example 3: Recommend products

Predict the top 10 products that customer `CustomerID=11176` is likely to order in the next 30 days.


In [ ]:
query = (
    'PREDICT LIST_DISTINCT(sales_order_details.ProductID, 0, 30, days) '
    'RANK TOP 10 FOR EACH customers.CustomerID'
)

result = client.relational(graph).predict(
    query,
    indices=[11176],
    run_mode='fast',
)
display(result)

How to interpret the result:

1. `ENTITY`: the customer with `CustomerID=11176`
2. `ANCHOR_TIMESTAMP`: the point in time from which the next 30 days are predicted
3. `CLASS`: a recommended `ProductID`
4. `SCORE`: a higher score indicates a stronger recommendation


### Example 4: Infer a missing product attribute

Predict the missing `Color` value for product `ProductID=870`.


In [ ]:
query = 'PREDICT products.Color FOR EACH products.ProductID'

result = client.relational(graph).predict(
    query,
    indices=[870],
    run_mode='fast',
)
display(result)

How to interpret the result:

1. `ENTITY`: the product with `ProductID=870`
2. `ANCHOR_TIMESTAMP`: the temporal context used for the prediction
3. `PREDICTION`: the predicted color category

Attribute inference can help enrich incomplete catalog records, but predicted values should be reviewed before being written back to a source system.


## Step 8. Close the client

Close the client when the notebook is finished to release its HTTP connection.


In [ ]:
client.close()